# 03 カメラとリアルタイム処理

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 緑の **✅ チェックポイント** が出たら次へ。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

## 3-1. 準備（カメラを開いたままにする）
このノートでは何度も撮るので、カメラを開いたまま使います。**最後に 3-9 の後始末セルを必ず実行**してください。

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
%matplotlib inline
from picamera2 import Picamera2
picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
print('カメラ開始')

## 3-2. 1 フレーム取得して処理
取得 → グレー → エッジ、を 1 枚で確認します。

In [ ]:
frame = picam.capture_array()
edges = cv2.Canny(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 80, 160)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title('原画')
plt.subplot(1,2,2); plt.imshow(edges, cmap='gray'); plt.axis('off'); plt.title('エッジ')
plt.show()

## 3-3. ライブ表示（パラパラ更新）
下のセルは、取得→処理→表示を繰り返して**動く映像**にします。`N` 枚で自動停止します。
（処理を `edges` から変えると、ライブの見え方が変わります）

In [ ]:
N = 40
for i in range(N):
    frame = picam.capture_array()
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    view  = cv2.Canny(gray, 80, 160)            # ← ここを変えると見え方が変わる
    clear_output(wait=True)
    plt.figure(figsize=(6,4)); plt.imshow(view, cmap='gray'); plt.axis('off')
    plt.title(f'live {i+1}/{N}'); plt.show()
print('ライブ終了')

## 3-4. 色で物体を追いかける（カラートラッキング）
**色のはっきりした物**（緑のペン等）を手に持って動かしてみましょう。重心に印が付きます。

In [ ]:
LOWER, UPPER = (35,80,80), (85,255,255)   # 緑。青なら100-130, 黄なら20-35
N = 40
for i in range(N):
    frame = picam.capture_array()
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER, UPPER)
    cnts,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    big = [c for c in cnts if cv2.contourArea(c) > 300]
    if big:
        c = max(big, key=cv2.contourArea); M = cv2.moments(c)
        cx, cy = int(M['m10']/M['m00']), int(M['m01']/M['m00'])
        cv2.circle(frame, (cx,cy), 12, (0,0,255), -1)
    clear_output(wait=True)
    plt.figure(figsize=(6,4)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'tracking {i+1}/{N}'); plt.show()
print('終了')

✅ **チェックポイント**：手に持った色の物に、赤い点が追従すれば成功です。

## 3-9. 後始末（必ず実行）
カメラを解放します。次のノート（04 AIカメラ）に進む前に実行してください。

In [ ]:
picam.close(); print('カメラ解放')